# Code agents with an allowlisted namespace

A complete, deterministic course companion.

## Learning objectives

- Treat generated code as an action that must cross a security boundary.
- Inspect an expression's syntax before evaluating it.
- Produce a deterministic three-course menu in a minimal namespace.

## Setup

The standard-library `ast` module provides the parser. The offline path has no package, model, network, or credential dependency.


For the optional live cell, install the dependency with `python -m pip install smolagents`.

## Complete implementation

Run the next cell to build the offline demonstration.

In [1]:
import ast

code_action = "[course for course in courses if guest_count >= 1]"
allowed_namespace = {
    "courses": ("starter", "main", "dessert"),
    "guest_count": 12,
}
allowed_node_types = (
    ast.Expression, ast.ListComp, ast.Name, ast.Load, ast.Store,
    ast.comprehension, ast.Compare, ast.GtE, ast.Constant,
)

parsed_action = ast.parse(code_action, mode="eval")
for node in ast.walk(parsed_action):
    if not isinstance(node, allowed_node_types):
        raise ValueError(f"Disallowed syntax: {type(node).__name__}")

referenced_names = {
    node.id for node in ast.walk(parsed_action)
    if isinstance(node, ast.Name) and isinstance(node.ctx, ast.Load)
}
if not referenced_names <= set(allowed_namespace) | {"course"}:
    raise ValueError("The action referenced a name outside the allowlist.")

menu = eval(
    compile(parsed_action, "<menu-action>", "eval"),
    {"__builtins__": {}},
    allowed_namespace,
)

## Deterministic assertions

These checks make the lesson reproducible without a model or network connection.

In [2]:
assert code_action == "[course for course in courses if guest_count >= 1]"
assert referenced_names == {"course", "courses", "guest_count"}
assert menu == ["starter", "main", "dessert"]

print("Offline assertions passed.")

Offline assertions passed.


## Results interpretation

The action can access only the explicitly supplied course data and guest count. Parsing, node validation, name validation, and an empty builtins mapping illustrate layered controls; production code should prefer a hardened sandbox for untrusted model output.

## Credentials

The secure offline demonstration needs none. The optional live cell checks `HF_TOKEN` without printing it.

In [3]:
import os

live_model_provider = os.getenv("LIVE_MODEL_PROVIDER", "huggingface").strip().casefold()
hf_token = os.getenv("HF_TOKEN")
if live_model_provider != "local" and not hf_token:
    print("SKIPPED: Set HF_TOKEN to run this optional smolagents example.")
else:
    try:
        from smolagents import CodeAgent
        if live_model_provider == "local":
            from smolagents import TransformersModel
        else:
            from smolagents import InferenceClientModel
    except ImportError:
        print("SKIPPED: smolagents is not installed; run `python -m pip install smolagents`.")
    else:
        if live_model_provider == "local":
            model = TransformersModel(
                model_id="Qwen/Qwen2.5-Coder-1.5B-Instruct",
                device_map="cpu",
                torch_dtype="auto",
                max_new_tokens=256,
            )
        else:
            model = InferenceClientModel(
                model_id="Qwen/Qwen2.5-Coder-32B-Instruct", token=hf_token
            )
        live_agent = CodeAgent(
            tools=[],
            model=model,
            max_steps=2,
            additional_authorized_imports=[],
        )
        live_result = live_agent.run("Return a Python list containing starter, main, and dessert.")
        print(live_result)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Return a Python list containing starter, main, and dessert.                                                     │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen2.5-Coder-1.5B-Instruct ──────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  meal_list = meal_plan()                                                                                          
  print(meal_list)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code execution failed at line 'meal_list = meal_plan()' due to: InterpreterError: Forbidden function evaluation: 
'meal_plan' is not among the explicitly allowed tools or defined/imported in the preceding code

... (4 outputs left out)

[Step 2: Duration 86.41 seconds| Input tokens: 4,290 | Output tokens: 244]

Reached max steps.

[Step 3: Duration 21.63 seconds| Input tokens: 4,782 | Output tokens: 291]

```python
# Starter: Salad
# Main: Grilled Chicken Salad
# Dessert: Chocolate Cake

meal_list = ["Salad", "Grilled Chicken Salad", "Chocolate Cake"]
print(meal_list)
```


## Completion checklist

- [ ] Parsed the code action before evaluation.
- [ ] Restricted syntax and names.
- [ ] Verified the deterministic menu.

## Attribution

Original example informed by the official [smolagents agent reference](https://huggingface.co/docs/smolagents/reference/agents) and [secure code execution guide](https://huggingface.co/docs/smolagents/tutorials/secure_code_execution). No official notebook is copied. Notebook code is released under the Apache-2.0 metadata license.